In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import logging
import os
import sys
import warnings
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, display  # noqa: E402
from sklearn.decomposition import FastICA  # noqa: E402
from sklearn.exceptions import ConvergenceWarning  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    load_joined_condition_wavelets,
    resolve_notebook_wavelet_cache_dir,
    resolve_wavelet_dir,
)
from src.analysis import iva_quality  # noqa: E402
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_topomaps,
    plot_participant_condition_topomaps,
)
from src.visualization.iva_quality_plots import (  # noqa: E402
    participant_sort_key,
    save_fig,
    topo_info_subset,
)
from src.visualization.jica_plots import (  # noqa: E402
    plot_global_tf_grid,
    plot_loading_bars,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Joint ICA of Wavelet Power — Channel Components, Both Conditions on the Recording Axis

The **ICA** counterpart of
[`wavelet_iva_channel_joined.ipynb`](../06-iva-condition-comparison/wavelet_iva_channel_joined.ipynb),
on the same cohort, the same wavelet caches and the same independence geometry — the
mixing dimension is **channels** and the sample axis is the joint **(frequency × time)**
plane. What changes is *how* the recordings are tied together: IVA gives every recording
its own unmixing matrix and couples them through the source-component vector, whereas
here **one** decomposition is fitted to all recordings at once by laying their channel
axes side by side.

```
Input:      (n_subjects, n_channels, n_freqs, n_times)   n_subjects = 2 × n_pairs
Stack:      (n_subjects × n_channels,  n_freqs × n_times)
            ─────── mixing ─────────   ─────  samples  ────
Transpose:  (n_freqs × n_times,  n_subjects × n_channels)   ← the FastICA input
FastICA over the stacked channel axis → N_ICA components
```

This is **joint ICA** (jICA): the feature axis is every recording's channels
concatenated, so an independent component is one shared spectro-temporal source
`(F, T)` together with a mixing column that spans *all* recordings — i.e. a
per-recording topography `(C,)` plus, in the size of that block, a per-recording
loading.

## Scope

Load, decompose, pin the sign and the order, and then draw the three things the
decomposition produces: the component **TF maps**, the per-recording **topographies**
as a Placebo/Psilocybin comparison, and the **per-recording loading** as a bar plot.
Nothing is scored, ranked or tested, and nothing is written to disk beyond the figures
and the reusable wavelet subset cache. The in-memory arrays are the input for whatever
comparison comes next.

## What jICA gives and what it costs, against the IVA variant

|  | jICA (this notebook) | [channel IVA](../06-iva-condition-comparison/wavelet_iva_channel_joined.ipynb) |
|---|---|---|
| Independence forced over | the (frequency × time) axis of the **joint** matrix | the same axis, per dataset, coupled across datasets by the SCV |
| Datasets | one — every recording's channels in one feature axis | one per recording |
| Sources | **one `(F, T)` map per component, shared by every recording** | one per (recording, component) |
| Mixing | one column per component, split into per-recording blocks | one matrix per recording |
| Sign ambiguity | **per component only** | per (recording, component) — needs resolving before any average |
| Condition contrast lives in | the **topographies and the loadings** | the sources *and* the patterns |

Two consequences are worth stating before any figure is read.

**There is no per-recording sign to resolve.** A component's sign flips its map and its
*whole* mixing column together, so a recording whose block comes out negative is
genuinely anti-phase relative to the rest — a result, not an ambiguity. That removes the
step the IVA notebooks spend two passes on (`Sigma_N`, then PC1 of the TF maps) and with
it the risk that arbitrary flips manufacture a condition difference.

**The price is that the conditions share one TF map per component.** The source axis is
common to the whole joint matrix, so this variant cannot show a Placebo-versus-Psilocybin
*time-frequency* difference at all: everything the conditions can differ in is in the
mixing column. That is exactly why the loading bar plot (Step 6) is not a diagnostic
here but the main read-out. For a genuine TF contrast, concatenate the conditions along
**time** instead —
[`wavelet_ica_channel_joined_tracks.ipynb`](wavelet_ica_channel_joined_tracks.ipynb),
the sibling notebook.

## Why the stacking is fair

`zscore_by_time` standardises every `(recording, channel, frequency)` time series to zero
mean and unit variance, so **every column of the stacked matrix has unit variance by
construction** (Step 1 checks it numerically). No recording can dominate the joint
whitening through sheer amplitude, and a difference in the loadings is therefore a
difference in how strongly that recording expresses the shared component, not a gain
artefact. The flip side, as everywhere in this project's wavelet workflows: an overall
power difference between the conditions is normalised away, so what is compared is
temporal and spectral *structure*.

## How the data is assembled

`load_joined_condition_wavelets` loads **each condition exactly as a single-condition
workflow does**, reusing the existing per-condition wavelet caches, then stacks the
participant-matched recordings on the subject axis (`pool_condition_subjects`). Nothing
upstream is re-run, re-aligned or re-transformed, and no `Joined_*` cache is written —
the pooled tensor exists only in memory. Only participants with a recording in *both*
conditions are kept, so the recording axis is a balanced within-participant design.

## Variables produced

| Variable | Shape | Description |
|----------|-------|-------------|
| `pooled` | — | `PooledConditionSubjects` — the pooled tensor plus its per-recording participant/condition bookkeeping |
| `bb_data` | `(S, C, F, T)` | Raw 4-D pooled wavelet power, `S = 2 × n_pairs` |
| `ica_input` | `(F·T, S·C)` | Z-scored samples × stacked-channel matrix — the FastICA input |
| `sources_ft` | `(F·T, K)` | Unit-variance independent sources |
| `tf_maps` | `(K, F, T)` | **Global** component TF maps, shared by every recording |
| `patterns` | `(S, K, C)` | Forward (mixing) channel topographies, per recording |
| `ic_variance` | `(K,)` | Share of the joint channel-space energy each component accounts for |
| `subject_ic_energy` | `(S, K)` | That share split by recording — the loading the bar plot draws |
| `retained` | float | Channel-space variance the whitening truncation kept |

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.ASSR
# The pooled dataset IS the condition here — the two real conditions below are stacked
# on the recording axis and the product is labelled Joined_<MusicType>.
CONDITION = ConditionVariants.JOINED
# Recording-axis block order: Placebo recordings first, then Psilocybin.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers. It must
# match the grid the cache was written with, or the cache is missed and recomputed.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

# ── Reuse / compute ──────────────────────────────────────────
# True is the intended setting: this notebook is built around reusing the two
# per-condition caches. False would recompute both from RAW_AFTER_ICA.
REUSE_WAVELETS = True

# ── Cohort, channel and time subset ────────────────────────────
# N_PAIRS_SUBSET counts *participants*, not recordings: each kept participant brings one
# recording per condition, so the pooled axis is 2 × N_PAIRS_SUBSET. Never slice the
# pooled recording axis by position — the leading rows are one whole condition block.
# Use ``pooled.select_participants`` (done in the loading cell).
N_PAIRS_SUBSET: int | None = 5
# Memory: the FastICA input is (F·T) × (S·C) float64. With the defaults below that is
# 50·3000 × 10·32 ≈ 384 MB; the z-scored tensor it is built from is another 384 MB
# (released straight after), and FastICA's whitening SVD works on a transposed copy, so
# budget roughly three times the input for the fit. Raising N_TIMES_SUBSET scales the
# sample axis linearly and raising N_CHANNELS_SUBSET or N_PAIRS_SUBSET the feature axis;
# keep both modest during exploration and raise them on a node with more RAM.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 3000  # first N time samples (F·T = 50 * this)

# ── ICA settings ──────────────────────────────────────────────
# N_ICA is the ONLY dimensionality knob. FastICA whitens the stacked channel matrix by
# SVD and keeps its leading N_ICA directions, so it performs the reduction itself and no
# separate PCA step is needed (the same argument as
# notebooks/04-wavelet-ica-analysis/separate_assr_wavelet_ica_channel.ipynb, which
# verified the two paths span an identical subspace).
#
# Unlike the IVA variant, N_ICA is NOT capped by the channel count: the feature axis is
# S·C, so there is far more room. What caps it in practice is convergence — on ASSR data
# FastICA reliably converges at small N_ICA and can fail at 10, and a non-converged
# unmixing is an arbitrary point on the solver's path rather than a fixed point. Step 2
# reports it; lower N_ICA (or try "deflation") if it does not converge.
N_ICA = 10
ICA_ALGORITHM = "parallel"  # or "deflation" — extracts one component at a time
ICA_FUN = "logcosh"  # contrast function: "logcosh", "exp" or "cube"
ICA_MAX_ITER = 2000
ICA_TOL = 1e-4
RANDOM_STATE = 42  # FastICA's initial unmixing is random; this pins it

# ── Figures ───────────────────────────────────────────────────
SAVE_PLOTS = True
# Which components the figures cover. None = every component.
COMPONENTS_TO_PLOT: list[int] | None = None
# Per-participant topography grids are one figure PER COMPONENT, so they are opt-in.
WRITE_PARTICIPANT_GRIDS = True
# Reference lines on the TF panels. The ASSR is continuous 40 Hz stimulation, so the
# stimulation frequency is the row worth locating on every map.
TF_FREQ_MARKS: list[float] = [iva_quality.ASSR_FREQ]
MARK_STIMULUS_ONSETS_ON_TF = True
# One colour per condition, shared by every panel so a bar never has to be looked up.
CONDITION_COLORS = {
    ConditionVariants.PLACEBO.value: "#0F6E8C",
    ConditionVariants.PSILOCYBIN.value: "#A6357F",
}

# ── Stimulus-locked epoch (Step 7) ────────────────────────────
# The paradigm window from src.definitions.constants.AssrEpoch, so this notebook, the
# 06 IVA notebooks and the 05 quality workflow all cut the SAME epoch: a short pre-onset
# baseline, then the stimulus plus an equally long post-stimulus interval.
# iva_quality.onset_window caps the post-onset span by the shortest inter-onset gap, so
# an epoch can never reach the next stimulus.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S
EPOCH_POST_S = AssrEpoch.POST_ONSET_S
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── Wavelet cache directories ─────────────────────────────────
# WAVELET_DIR is the source of truth: one big compressed file per condition spanning the
# full cohort, all 195 channels and the whole aligned time axis (tens of GB each for
# ASSR). WAVELET_SUBSET_CACHE_DIR holds per-extent copies under the stage-03 notebook
# that owns the Morlet transform; it is keyed by condition, frequency grid and extent —
# not by the notebook that wrote it — so the entries this notebook fills in are the same
# ones the 03/04/05/06 workflows read, and vice versa.
WAVELET_DIR: Path = resolve_wavelet_dir(None, EXPERIMENT_NAME) / "broadband"
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME) / "broadband"
)
REUSE_WAVELET_SUBSET_CACHE = True

# ── Plots directory ───────────────────────────────────────────
# Canonical notebook layout: plots/<experiment>/<broadband|bands>/<analysis_type>/,
# with ica_<n> isolating sweeps over N_ICA.
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "07-ica-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "ica_channel_joined"
    / f"ica_{N_ICA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment / group  : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Conditions pooled   : {[c.value for c in CONDITIONS_TO_POOL]} (recording axis)")
print(f"Wavelet source cache: {WAVELET_DIR}")
print(
    f"Wavelet subset cache: {WAVELET_SUBSET_CACHE_DIR}  "
    f"(reuse: {REUSE_WAVELET_SUBSET_CACHE})"
)
print(f"Frequencies         : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(
    f"Decomposition       : {N_ICA} ICs over {ICA_ALGORITHM}/{ICA_FUN} "
    f"(FastICA whitens the stacked channel axis itself; subset extents only), "
    f"seed {RANDOM_STATE}"
)
print(f"Plots directory     : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")
print(
    f"Stimulus epoch      : [-{EPOCH_PRE_S}, {EPOCH_POST_S}] s around each onset "
    f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
    f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)"
)

## Data Loading — both conditions on one recording axis

One call does the whole join: load each condition's wavelet power from its existing
cache, keep the participants present in both, and stack them on the recording axis.

**First run vs. every run after.** The source-of-truth wavelet tensor is decompressed in
full before it is trimmed to the requested channel/time extent (the stimulus alignment is
defined over the whole cohort, so that cache cannot be written per subset) — tens of GB
per condition for ASSR. The two conditions are loaded and trimmed **one at a time**, so
the peak is one condition's cache, the same peak as a single-condition notebook, not both
at once. Run the first pass on a node sized for that.

The trimmed result is then written to `WAVELET_SUBSET_CACHE_DIR`, and later runs at the
same extent read it back without opening the big cache at all. Changing
`N_CHANNELS_SUBSET` or `N_TIMES_SUBSET` asks for a different extent and pays the full
cost once more, for that extent.

In [ ]:
pooled, condition_analyzers = load_joined_condition_wavelets(
    MUSIC_TYPE,
    EXCLUSION_CATEGORIES,
    FREQS,
    wavelet_dir=WAVELET_DIR,
    experiment_name=EXPERIMENT_NAME,
    representation=REPRESENTATION,
    conditions=CONDITIONS_TO_POOL,
    n_channels=N_CHANNELS_SUBSET,
    n_times=N_TIMES_SUBSET,
    reuse_wavelets=REUSE_WAVELETS,
    subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
    reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
)

print(f"Pooled dataset : {pooled.data.label}")
print(f"Participants   : {pooled.n_pairs}  ->  {pooled.n_subjects} recordings")
print(
    f"Shape          : {pooled.data.data.shape}  "
    "(recordings × channels × freqs × times)"
)

# Restrict the cohort by *participant*, so both of a participant's recordings are kept
# and the pooled axis stays balanced.
if N_PAIRS_SUBSET is not None:
    keep = sorted(set(pooled.participants))[:N_PAIRS_SUBSET]
    pooled = pooled.select_participants(keep)
    print(f"\nUsing first {len(keep)} participant(s): {keep}")
    print(f"Shape          : {pooled.data.data.shape}")

print(f"\nRecording axis : {list(pooled.subject_labels)}")

## Dataset Selection

One dataset — the pooled one — so this cell unpacks it into the names the other variants
use (`bb_data`, `sfreq`, `n_subjects`, …), records the per-recording condition
bookkeeping, builds the topomap `Info`, and names the two shared figures this notebook
uses beyond
[`src/visualization/iva_condition_plots.py`](../../src/visualization/iva_condition_plots.py):

Both come from
[`src/visualization/jica_plots.py`](../../src/visualization/jica_plots.py) rather than
being defined here, so this notebook and
[`scripts/run_wavelet_jica.py`](../../scripts/run_wavelet_jica.py) draw the identical
figure from the identical code.

- `plot_global_tf_grid` — the component TF maps. Deliberately **not**
  `plot_condition_mean_tf_maps`: there is one map per component here, shared by every
  recording, so there is no recording axis to average over and no per-recording gain to
  equalise. Equalising is the first thing that function does, and applied to a single map
  per row it would rescale each row by its own amplitude — normalising away exactly the
  between-row difference a grid is drawn to show.
- `plot_loading_bars` — the per-recording loading, grouped by participant and coloured by
  condition.

In [ ]:
LABEL = pooled.data.label

bb_ad = pooled.data
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

# Per-recording bookkeeping: which participant and which condition each row is.
subject_participants = list(pooled.participants)
subject_conditions = [c.value for c in pooled.subject_conditions]
subject_labels = list(pooled.subject_labels)
condition_masks = {c: pooled.condition_mask(c) for c in pooled.conditions}
CONDITION_ROWS = [c.value for c in pooled.conditions]
# Row label for anything the conditions SHARE — here the TF maps, by construction.
SHARED_ROW = " + ".join(CONDITION_ROWS) + " (shared)"
# What pins the component sign, named on every figure. One sign per component, not per
# recording: jICA has no per-recording sign ambiguity to resolve.
SIGN_NOTE = "largest |TF| excursion positive (one sign per component)"

print(f"Dataset      : {LABEL}")
print(f"Shape        : {bb_data.shape}  (recordings × channels × freqs × times)")
print(f"Duration     : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range   : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
for condition, mask in condition_masks.items():
    print(
        f"  {condition.value:<12}: {int(mask.sum())} recording(s) at rows "
        f"{np.flatnonzero(mask).tolist()}"
    )

# ── Stimulus onset markers ────────────────────────────────────
# Both conditions share the aligned time base, so one set of onsets describes every
# recording; the first pooled condition's is taken as the reference. ``None`` for
# experiments without stimulus annotations (e.g. PSILO_MUSIC) -> no markers.
_onset_samples = pooled.condition_onsets(pooled.conditions[0])
if _onset_samples is None:
    stimulus_onset_times = np.array([])
else:
    stimulus_onset_times = _onset_samples[_onset_samples < n_times] / sfreq
print(f"Stimulus onsets in window : {len(stimulus_onset_times)}")

# ── Topomap layout ───────────────────────────────────────────
# The analyser's Info, restricted to the notebook's channel subset in the same order as
# the decomposition's channel axis. Both conditions were preprocessed onto the same
# montage, so either analyser will do.
ica_info = topo_info_subset(condition_analyzers[pooled.conditions[0]].info, n_channels)
print(
    f"Topomap channels : {len(ica_info['ch_names'])} "
    f"(first={ica_info['ch_names'][0]}, last={ica_info['ch_names'][-1]})"
)


# ── Figure helpers ───────────────────────────────────────────
# The two figures this notebook needs beyond
# src/visualization/iva_condition_plots.py live in
# src/visualization/jica_plots.py, imported above rather than defined here so the
# notebook and scripts/run_wavelet_jica.py draw the identical figure:
#
#   plot_global_tf_grid  the component TF maps. Deliberately NOT
#       plot_condition_mean_tf_maps: there is one map per component here, shared by
#       every recording, so there is no recording axis to average over — and that
#       function's first step is equalize_subject_influence, which with one map per
#       row would rescale each row by its own amplitude and normalise away exactly
#       the between-row difference a grid is drawn to show.
#   plot_loading_bars    the per-recording loading, grouped by participant.
#
# DIFFERENCE_ROW is the row label that gets its own colour limit, since a
# difference is far weaker than the maps it comes from.


def _save(fig, name: str) -> None:
    """Write *fig* into PLOTS_DIR under the band prefix, when saving is on."""
    if not SAVE_PLOTS:
        return
    path = PLOTS_DIR / f"{name}.png"
    save_fig(fig, path)
    print(f"saved {path}")

---
## Step 1 — Z-score, Then Lay the Recordings' Channel Axes Side by Side

**Z-scoring** normalises each `(recording, channel, frequency)` time series to zero mean
and unit variance over time. Wavelet power is ~1/f, so without it the low-frequency,
high-power samples would dominate and "the shared spectrum" would collapse to "the shared
low frequencies". It is applied identically in the IVA variants, and — because every
series is normalised independently — applying it before or after the pooling gives the
same array.

Two consequences, both load-bearing here:

- **Every column of the stacked matrix has unit variance**, so no recording can dominate
  the joint whitening by amplitude. The cell checks it rather than asserting it in prose.
- **An overall power difference between the conditions is normalised away**, so what the
  decomposition compares is temporal and spectral *structure*, not amplitude.

**The stack** is the one thing that differs from the IVA variant. Each recording's
`(C, F, T)` slice is flattened to `(C, F·T)` exactly as there, but instead of becoming its
own dataset the `S` blocks are stacked into a single `(S·C, F·T)` matrix, then transposed
so FastICA sees `(F·T, S·C)`: **samples = (frequency, time) bins, mixing variables = every
recording's channels**. The flatten keeps frequency slow and time fast
(`index = f·T + t`), so the sample axis reshapes cleanly back to `(F, T)` afterwards, and
the row index is `s·C + c`, so the mixing column reshapes cleanly back to
`(S, C)`.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power |
| `ica_input` | `(F·T, S·C)` | Samples × stacked channels — the FastICA input |

In [ ]:
# Z-score along time: each (recording, channel, frequency) slice → mean 0, std 1.
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Stack: (S, C, F, T) → (S*C, F*T) → transpose to samples × mixing. bb_z is
# C-contiguous, so the flattened sample axis runs frequency-slow / time-fast
# (index = f*T + t) and the feature axis runs recording-slow / channel-fast
# (index = s*C + c). Both reshape back exactly after the decomposition.
n_samples_ft = n_freqs * n_times
n_features = n_subjects * n_channels
ica_input = np.ascontiguousarray(bb_z.reshape(n_features, n_samples_ft).T)

print(f"Stacked reshape : {ica_input.shape}  (F*T samples, S*C mixing variables)")
print(f"  Samples       : {n_samples_ft}  (F={n_freqs} * T={n_times})")
print(f"  Mixing dim    : {n_features}  (S={n_subjects} recordings * C={n_channels})")
print(f"  Memory        : {ica_input.nbytes / 1e9:.2f} GB")

# Every column must have unit variance — that is what makes the stacking fair, so it is
# checked rather than claimed. Deviations come only from the guard zscore_by_time applies
# to constant series.
_col_std = ica_input.std(axis=0)
print(
    f"\nColumn std      : min {_col_std.min():.4f}, max {_col_std.max():.4f} "
    "(1.0 = no recording can dominate the whitening by gain)"
)
_worst = float(np.abs(_col_std - 1.0).max())
if _worst > 1e-6:
    print(
        f"  NOTE: worst deviation {_worst:.2e} — some (recording, channel, frequency) "
        "series was constant in time and left at its guard value."
    )

# The z-scored tensor is only needed to build the matrix; release it (bb_data, the raw
# pooled tensor, stays alive inside ``pooled``).
del bb_z

---
## Step 2 — One Joint FastICA Over the Stacked Channel Axis

The whole decomposition is this one fit. FastICA unmixes the stacked channel axis
directly, and because the sample axis is the joint (frequency × time) plane, the
components come out as spectro-temporal sources.

**`FastICA` does the channel reduction itself here, and that is why this notebook is
capped to a subset extent.** `FastICA(n_components=K)` whitens the matrix by SVD and keeps
its leading `K` directions — the same subspace a `PCA(K)` would have produced — but it
gets there by calling `scipy.linalg.svd`, and LAPACK indexes with **32-bit integers**. A
matrix with more than `2**31 - 1` elements is refused outright:

```
ValueError: Indexing a matrix of 10993320000 elements would incur an in integer
overflow in LAPACK.
```

At the full ASSR extent (12 participants × 195 channels = 2340 features) that ceiling is
~918k samples, i.e. ~73 s of both tracks on a 50-bin frequency grid. This is a hard limit,
not a memory budget: no node size makes the call work.

The cell below therefore **refuses** rather than failing deep inside sklearn, and says
where to go instead. For the full recording use
[`scripts/run_wavelet_jica.py`](../../scripts/run_wavelet_jica.py), whose
[`fit_joint_ica`](../../src/analysis/wavelet_jica.py) takes the leading directions from
the `(features, features)` covariance — 2340 × 2340, trivial — and runs FastICA on the
scores. The two routes span the *same* subspace (verified equal to 4e-16), so this
notebook and that script describe the same decomposition; only the arithmetic that gets
there differs. (This is the finding of
[`separate_assr_wavelet_ica_channel.ipynb`](../04-wavelet-ica-analysis/separate_assr_wavelet_ica_channel.ipynb),
which verified the two paths span an identical subspace to 14 significant figures. Note
that the IVA variants *do* need their explicit per-recording PCA, for a different reason:
`iva_g` requires a square mixing matrix per dataset.)

Four outputs deserve attention.

**`tf_maps`** `(K, F, T)` — each component's score map, reshaped back to `(F, T)`.
FastICA's `unit-variance` whitening fixes every source to unit variance, so the maps share
a scale and one colour limit across components would be fair; the amplitude lives in the
pattern instead. **There is one map per component, not one per recording** — the defining
property of this variant.

**`patterns`** `(S, K, C)` — the **forward (mixing)** channel patterns, `ica.mixing_.T`
reshaped to split the stacked feature axis back into recordings. This is what belongs on a
topomap. The unmixing rows (`components_`) are spatial *filters* — a different object, and
plotting them instead is the classic filter-vs-pattern error (Haufe et al., 2014,
NeuroImage 87:96-110); MNE follows the same convention, `ica.get_components()` returns the
mixing matrix. Because `components_` carries the whitening folded in, `mixing_ =
pinv(components_)` undoes it too.

**`retained`** — the fraction of the joint channel-space variance the whitening truncation
kept. It is the hard ceiling on everything the components can carry: whatever is orthogonal
to the retained subspace is gone. It is the analogue of a PCA's
`explained_variance_ratio_.sum()` and equals it exactly.

**`ic_variance`** `(K,)` and **`subject_ic_energy`** `(S, K)` — the energy of each
component's rank-one back-projection over the total sum of squares, and that energy split
by recording. `sources @ patterns` reconstructs the retained part of the centred matrix,
so these are the energies of its rank-one terms, and the per-recording split is exact
because the feature axis partitions by recording. A unit-variance source says nothing
about how much of the data a component accounts for; this is what does, and its
per-recording split is the loading Step 6 draws.

Non-convergence is captured rather than printed and lost: a non-converged unmixing is
wherever the solver stopped, not a fixed point.

In [ ]:
def fit_joint_channel_ica(matrix, n_ica, seed, chunk=20_000):
    """Fit one FastICA over the stacked channel axis of the pooled tensor.

    :param matrix: ``(F*T, S*C)`` samples × stacked-channels matrix.
    :param n_ica: Independent components to extract. Also sets the whitening
        truncation, since FastICA reduces the feature axis itself.
    :raises ValueError: If *matrix* has more elements than 32-bit LAPACK can index,
        which is a hard ceiling rather than a memory limit — see the markdown above
        and use scripts/run_wavelet_jica.py for the full extent.
    :param seed: ``random_state`` for FastICA's initial unmixing.
    :param chunk: Sample rows per block when the sums of squares are accumulated. The
        centred matrix and the back-projection are each the size of *matrix*, so
        forming them whole would triple the peak for two scalars; chunking keeps it at
        a few tens of MB and gives the identical result.
    :return: ``(sources, patterns_stacked, ic_variance, feature_shares, retained,
        total_ss, converged)`` with shapes ``(F*T, K)``, ``(K, S*C)``, ``(K,)``,
        ``(K, S*C)``, float, float, bool. *feature_shares* is each feature's rank-one
        energy share, ready to be pooled per recording.
    """
    # scipy.linalg.svd, which FastICA's whitening calls, indexes with 32-bit ints.
    # Refuse here with something actionable rather than failing deep inside sklearn.
    if matrix.size > np.iinfo(np.int32).max:
        raise ValueError(
            f"The stacked matrix has {matrix.size:,} elements, above the "
            f"{np.iinfo(np.int32).max:,} that 32-bit LAPACK can index, so FastICA's "
            "whitening SVD cannot run on it at any node size. Lower N_TIMES_SUBSET / "
            "N_CHANNELS_SUBSET / N_PAIRS_SUBSET for this notebook, or use "
            "scripts/run_wavelet_jica.py, which reduces the channel axis through the "
            "(features, features) covariance instead and has no such ceiling."
        )
    ica = FastICA(
        n_components=n_ica,
        algorithm=ICA_ALGORITHM,
        fun=ICA_FUN,
        whiten="unit-variance",
        random_state=seed,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOL,
    )
    # Capture ConvergenceWarning instead of letting it print once and vanish.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        sources = ica.fit_transform(matrix)  # (F*T, K)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)

    # Forward model, directly in stacked-channel space.
    patterns_stacked = ica.mixing_.T  # (K, S*C)

    # Total and residual sums of squares, accumulated in blocks of samples so neither
    # the centred matrix nor the back-projection is ever formed whole.
    feature_mean = matrix.mean(axis=0)
    total_ss = 0.0
    residual_ss = 0.0
    for start in range(0, matrix.shape[0], chunk):
        block = matrix[start : start + chunk] - feature_mean
        total_ss += float((block**2).sum())
        block -= sources[start : start + chunk] @ patterns_stacked
        residual_ss += float((block**2).sum())
    if total_ss <= 0.0:
        nan_k = np.full(n_ica, np.nan)
        return (
            sources,
            patterns_stacked,
            nan_k,
            np.full_like(patterns_stacked, np.nan),
            np.nan,
            total_ss,
            converged,
        )
    retained = 1.0 - residual_ss / total_ss
    source_ss = (sources**2).sum(axis=0)  # (K,)
    # Rank-one energy, per feature. Summing over a recording's channel block gives that
    # recording's share; summing over everything gives ic_variance.
    subject_shares = source_ss[:, np.newaxis] * patterns_stacked**2 / total_ss
    ic_variance = subject_shares.sum(axis=1)
    return (
        sources,
        patterns_stacked,
        ic_variance,
        subject_shares,
        retained,
        total_ss,
        converged,
    )


(
    sources_ft,
    patterns_stacked,
    ic_variance,
    _feature_shares,
    retained,
    total_ss,
    converged,
) = fit_joint_channel_ica(ica_input, N_ICA, RANDOM_STATE)

# Fold both axes back: the samples into (F, T), the stacked features into (S, C).
tf_maps = sources_ft.T.reshape(N_ICA, n_freqs, n_times)  # (K, F, T) — GLOBAL
patterns = patterns_stacked.reshape(N_ICA, n_subjects, n_channels).transpose(1, 0, 2)
subject_ic_energy = (
    _feature_shares.reshape(N_ICA, n_subjects, n_channels).sum(axis=2).T
)  # (S, K)

print(f"Sources (F*T, K)      : {sources_ft.shape}")
print(f"TF maps (K, F, T)     : {tf_maps.shape}  — one map per component, SHARED")
print(f"Patterns (S, K, C)    : {patterns.shape}  — one topography per recording")
print(f"Loading (S, K)        : {subject_ic_energy.shape}")
print(
    f"\nRetained variance     : {retained * 100:.1f}% of the joint channel space "
    f"({N_ICA} of {n_features} directions)"
)
print(
    f"ic_variance           : top {ic_variance.max() * 100:.2f}%, "
    f"sum {ic_variance.sum() * 100:.2f}%"
)
print(f"FastICA converged     : {converged} ({ICA_ALGORITHM}/{ICA_FUN})")
if not converged:
    print(
        f"  WARNING: FastICA did not converge within {ICA_MAX_ITER} iterations at "
        f"tol={ICA_TOL:g}, so the unmixing is wherever the solver stopped rather than a "
        "fixed point. Lower N_ICA, or try ICA_ALGORITHM='deflation'. On ASSR data "
        "raising ICA_MAX_ITER tends to make the result drift further rather than settle."
    )

# The per-recording split must add up to the component's total, by construction — a
# mismatch would mean the feature axis was folded back in the wrong order.
np.testing.assert_allclose(
    subject_ic_energy.sum(axis=0),
    ic_variance,
    rtol=1e-9,
    atol=1e-12,
    err_msg="The per-recording energy split does not sum to ic_variance; the stacked "
    "feature axis was reshaped in the wrong order.",
)
print("\nPer-recording energy split sums to ic_variance.")

---
## Step 3 — Sign and Order

FastICA fixes neither the sign nor the order of its components, so both have to be pinned
before plotting — otherwise the same data replotted with a different seed looks like a
different result. Neither choice is a claim about the data:

- **Sign.** `(map, pattern)` and `(−map, −pattern)` are the *same* component, so the sign
  is free. Convention here, matching
  [`separate_assr_wavelet_ica_channel.ipynb`](../04-wavelet-ica-analysis/separate_assr_wavelet_ica_channel.ipynb):
  flip each component so the **largest absolute excursion of its TF map is positive**, so
  each panel's dominant event reads as a power *increase*. It is deliberately
  hypothesis-free — it says nothing about 40 Hz — so it stays valid whatever is measured
  next.
- **Order.** FastICA's output order is meaningless, so components are sorted by
  `ic_variance` descending: `IC 1` accounts for the largest share of the joint
  channel-space energy. That is a statement about *size*, not relevance — a 40 Hz steady
  state can easily sit below a broad onset response, so read all of them rather than the
  first few.

**One sign for the whole component, and that is the point.** In the IVA variants a sign
belongs to a `(recording, component)` pair, which is why they spend two alignment passes
resolving it and why an unresolved flip there can manufacture a condition difference out of
nothing. Here the sign multiplies the map and the *entire* mixing column together, so a
recording whose block comes out negative is genuinely inverted relative to the rest. That
is a finding, and it survives into the topographies below.

Both operations leave the back-projection `Σ_k source_k ⊗ pattern_k` untouched, which is
what the assertion checks: flipping and reordering rearrange the bookkeeping, not the
decomposition.

In [ ]:
def back_projection(tf, pat, rows):
    """``Σ_k source_k ⊗ pattern_k`` on selected sample rows — sign- and order-invariant.

    Only *rows* are reconstructed: the full back-projection is the size of the whole
    input matrix, and a random few thousand samples settle the question just as well.

    :param tf: ``(K, F, T)`` global component maps.
    :param pat: ``(S, K, C)`` per-recording forward patterns.
    :param rows: Sample indices into the flattened ``F*T`` axis.
    :return: ``(len(rows), S*C)`` reconstruction of those rows of the stacked matrix.
    """
    n_k = tf.shape[0]
    return tf.reshape(n_k, -1)[:, rows].T @ pat.transpose(1, 0, 2).reshape(n_k, -1)


_check_rows = np.random.default_rng(RANDOM_STATE).choice(
    n_samples_ft, size=min(2000, n_samples_ft), replace=False
)
_recon_before = back_projection(tf_maps, patterns, _check_rows)

# ── Sign: make each component's largest absolute TF excursion positive ─────────
_flat = tf_maps.reshape(N_ICA, -1)
_peak = np.take_along_axis(_flat, np.abs(_flat).argmax(axis=1)[:, np.newaxis], axis=1)
signs = np.where(_peak[:, 0] < 0.0, -1.0, 1.0)  # (K,)
tf_maps = tf_maps * signs[:, np.newaxis, np.newaxis]
patterns = patterns * signs[np.newaxis, :, np.newaxis]

# ── Order: largest share of the joint channel-space energy first ───────────────
order = np.argsort(-ic_variance, kind="stable")
tf_maps = tf_maps[order]
patterns = patterns[:, order]
ic_variance = ic_variance[order]
subject_ic_energy = subject_ic_energy[:, order]
sources_ft = sources_ft[:, order] * signs[order][np.newaxis, :]

# Neither the flip nor the reordering may change what the components add up to.
np.testing.assert_allclose(
    back_projection(tf_maps, patterns, _check_rows),
    _recon_before,
    atol=1e-9,
    err_msg="Re-orienting or reordering changed the back-projection — the sign was not "
    "applied to the map and the patterns together.",
)
del _recon_before

COMP_INDICES = list(range(N_ICA)) if COMPONENTS_TO_PLOT is None else COMPONENTS_TO_PLOT
_out_of_range = [k + 1 for k in COMP_INDICES if not 0 <= k < N_ICA]
if _out_of_range:
    raise ValueError(
        f"COMPONENTS_TO_PLOT names IC {_out_of_range}, outside 1..{N_ICA}."
    )

print(
    f"Flipped {int((signs < 0).sum())}/{signs.size} component(s); reordered by "
    "ic_variance. Back-projection unchanged."
)
print("\n  IC   ic_variance   loading spread across recordings (min–max % of IC)")
for k in range(N_ICA):
    _share = subject_ic_energy[:, k] / subject_ic_energy[:, k].sum()
    print(
        f"  {k + 1:>3}   {ic_variance[k] * 100:9.2f}%   "
        f"{_share.min() * 100:5.1f} – {_share.max() * 100:5.1f}   "
        f"(even split would be {100 / n_subjects:.1f})"
    )
print(f"\nComponents in the figures: {[k + 1 for k in COMP_INDICES]}")

---
## Step 4 — The Component TF Maps

One row, because there is one map per component and both conditions share it. Drawing it
as a two-row condition comparison would produce two identical rows and an all-zero
difference — a figure that looks like a null result but is really a statement about the
model.

That is not a defect of jICA, it is its geometry: the sample axis is common to the joint
matrix, so the conditions cannot differ *here*. They differ in the mixing column, which is
Step 5 and Step 6. For a genuine time-frequency contrast, concatenate the conditions along
time instead —
[`wavelet_ica_channel_joined_tracks.ipynb`](wavelet_ica_channel_joined_tracks.ipynb).

Each column is on its own symmetric limit (printed in the title), because a common limit
across components would render the weaker ones flat. The green line is the 40 Hz
stimulation frequency; the faint vertical lines are the stimulus onsets, drawn only when
there are few enough not to wash the map out.

In [ ]:
fig_tf = plot_global_tf_grid(
    {SHARED_ROW: tf_maps},
    [SHARED_ROW],
    COMP_INDICES,
    FREQS,
    time,
    label=LABEL,
    title="Component TF maps (one per component, shared by both conditions)",
    time_marks=stimulus_onset_times if MARK_STIMULUS_ONSETS_ON_TF else None,
    freq_marks=TF_FREQ_MARKS,
    sign_note=SIGN_NOTE,
    save_path=(PLOTS_DIR / "shared_tf_maps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")
if SAVE_PLOTS:
    print(f"Saved to {PLOTS_DIR}")

---
## Step 5 — Condition-Mean Topographies per Component

The first half of the contrast. Every recording has its own block of the mixing column, so
there **is** a per-condition topography here, and this is the standard grid: rows =
conditions, columns = components, plus a **difference** row (Psilocybin − Placebo), because
a small difference between two similar maps is far easier to see drawn directly than
inferred from the two panels above it.

Scaling rules, from
[`iva_condition_plots`](../../src/visualization/iva_condition_plots.py) so these figures
read exactly like the IVA ones: one symmetric limit per column shared by the condition
rows, no limit shared across columns, the difference row on its own limit per column.

**Every recording is put on a common scale first** (`equalize_subject_influence`), which
is the right thing here and worth being explicit about: it makes the grid a comparison of
topographic *shape*, and hands the amplitude question — how strongly each recording
expresses the component — to the loading bar plot in Step 6, where it is the whole point.
Without the rescaling the loudest few recordings would set the colour limit *and* dominate
both condition means.

The per-participant grids that follow (`WRITE_PARTICIPANT_GRIDS`) put Placebo on the first
row and Psilocybin on the second, one column per participant ordered by ID, so a
participant's two recordings sit directly one above the other. That vertical pairing is
what answers the question the means cannot: is a difference in the means shared across the
group, or carried by one or two people?

In [ ]:
fig_topo = plot_condition_mean_topomaps(
    patterns,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    ica_info,
    n_channels,
    COMP_INDICES,
    label=LABEL,
    alignment_note=SIGN_NOTE,
    save_path=(PLOTS_DIR / "condition_mean_topomaps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

if WRITE_PARTICIPANT_GRIDS:
    topo_paths = plot_participant_condition_topomaps(
        patterns,
        subject_participants,
        subject_conditions,
        CONDITION_ROWS,
        ica_info,
        n_channels,
        COMP_INDICES,
        label=LABEL,
        root_dir=PLOTS_DIR / "participants",
        alignment_note=SIGN_NOTE,
    )
    print(
        f"Wrote {len(topo_paths)} per-participant topography figure(s) to "
        f"{PLOTS_DIR / 'participants'}"
    )
    for path in topo_paths:
        display(Image(filename=str(path)))

---
## Step 6 — Each Recording's Contribution to the Loading

The second half of the contrast, and in this variant the one that carries it. Because the
TF maps are shared, everything psilocybin could change about a component is in the mixing
column — its shape (Step 5) and its **size** per recording, which is this.

Two normalisations of the same quantity, because they answer different questions:

- **Share of the joint channel-space energy** (`subject_ic_energy`) — recording *s*'s
  rank-one energy on component *k*, as a percentage of the total sum of squares. Comparable
  across components *and* recordings: it says how much of the whole dataset that
  (recording, component) pair accounts for. This is the one to read for "does psilocybin
  weaken this component".
- **Share of the component** (each column normalised to sum to 100%) — how component *k*'s
  energy is divided among the recordings. This says whether a component is a group
  phenomenon or one person's: an even split over `S` recordings would be
  `100/S` % each, and a component sitting almost entirely on one or two bars is not
  a group result whatever its condition difference looks like.

Because the sources are shared and unit-variance, a recording's energy share is exactly
proportional to the **squared L2 norm of its block** of the mixing column, so this is the
loading in the ordinary sense — no separate measure is needed, and the bars are directly
comparable across recordings.

Two things to keep in mind while reading them. The bars are **paired**: each participant's
Placebo and Psilocybin bars come from the same person, so the within-participant
consistency of the difference matters more than the group means. And a bar is a
*magnitude*: a recording whose topography is inverted relative to the group (Step 3) still
gets a tall bar, so read the two figures together.

No test is run here. The paired table printed below is the input a paired test would take.

In [ ]:
# ── Absolute: share of the joint channel-space energy ─────────────────────────
fig_load = plot_loading_bars(
    subject_ic_energy * 100.0,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    COMP_INDICES,
    label=LABEL,
    title="Per-recording component loading",
    ylabel="% of joint channel-space energy",
    colors=CONDITION_COLORS,
    save_path=(PLOTS_DIR / "loading_bars_absolute.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

# ── Relative: how each component's energy is divided among the recordings ─────
within_component = subject_ic_energy / subject_ic_energy.sum(axis=0, keepdims=True)
fig_load_rel = plot_loading_bars(
    within_component * 100.0,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    COMP_INDICES,
    label=LABEL,
    title=(
        f"Share of each component carried by each recording "
        f"(even split = {100 / n_subjects:.1f}%)"
    ),
    ylabel="% of this IC's energy",
    colors=CONDITION_COLORS,
    save_path=(PLOTS_DIR / "loading_bars_within_component.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

# ── The paired read-out, as a table ───────────────────────────────────────────
# One row per (participant, component): the two conditions' loadings and their
# difference. Nothing is tested — this is the input a paired test would take.
_rows = []
_participants_sorted = sorted(set(subject_participants), key=participant_sort_key)
_row_of = {
    (participant, condition): s
    for s, (participant, condition) in enumerate(
        zip(subject_participants, subject_conditions)
    )
}
for k in COMP_INDICES:
    for participant in _participants_sorted:
        entry = {"IC": k + 1, "participant": participant}
        for condition in CONDITION_ROWS:
            s = _row_of.get((participant, condition))
            entry[condition] = np.nan if s is None else subject_ic_energy[s, k] * 100.0
        entry["difference"] = entry[CONDITION_ROWS[0]] - entry[CONDITION_ROWS[1]]
        _rows.append(entry)
loading_frame = pd.DataFrame(_rows)
print(
    f"Loading table: % of joint channel-space energy, difference = "
    f"{CONDITION_ROWS[0]} − {CONDITION_ROWS[1]}"
)
display(loading_frame.round(3))

print("\nPer-component summary (mean over participants):")
display(
    loading_frame.groupby("IC")[[*CONDITION_ROWS, "difference"]]
    .agg(["mean", "std"])
    .round(3)
)

---
## Step 7 — Stimulus-Locked TF Maps (Averaged Over Stimuli)

Step 4 draws each component's TF map over the **whole** recording. That view answers "what
does this component do", but it cannot answer "what does it do *to a stimulus*": the ASSR
is a train of short stimuli, and a response locked to their onsets is smeared across the
whole map when the map spans every stimulus at once.

So this is the second view of the same maps: **cut a fixed epoch around every stimulus
onset and average it**. What was a `(F, T)` whole-recording map becomes a `(F, W)`
onset-averaged one.

The epoch is the **paradigm window** from
[`AssrEpoch`](../../src/definitions/constants.py), via `iva_quality.onset_window` — the
same call the 05 and 06 workflows make, so no onset-locked ASSR figure in this project is
cut on a different window than another: a short interval before each onset
(`EPOCH_PRE_S` = 0.1 s) as the baseline, then `EPOCH_POST_S` = 1.0 s after it (the 0.5 s
stimulus plus 0.5 s of post-stimulus, so a response that outlasts the stimulus is still
visible), capped by the shortest inter-onset gap so an epoch can never reach the next
stimulus.

**No baseline subtraction, deliberately.** Step 1 already zeroed each `(recording, channel,
frequency)` series' time-mean, so the pre-onset interval reads ≈ 0 by construction and
subtracting it would only add noise. That makes the pre-onset interval worth *looking* at:
it is where the map should be flat, so structure there is a warning sign, not a response.
The prominent black lines are the stimulus **onset** (dashed, `t = 0`) and its **offset**
(dotted).

Skipped for experiments without stimulus annotations (e.g. PSILO_MUSIC), and for a time
subset too short to fit `MIN_ONSETS_FOR_EPOCH_AVERAGE` epochs.

In [ ]:
onset_samples_in = (
    np.array([], dtype=int)
    if _onset_samples is None
    else _onset_samples[_onset_samples < n_times].astype(int)
)

if onset_samples_in.size:
    EPOCH_PRE, EPOCH_POST = iva_quality.onset_window(onset_samples_in, n_times, sfreq)
    n_fitting = int(
        (
            (onset_samples_in - EPOCH_PRE >= 0)
            & (onset_samples_in + EPOCH_POST <= n_times)
        ).sum()
    )
else:
    EPOCH_PRE = EPOCH_POST = 0
    n_fitting = 0

run_epoch_view = n_fitting >= MIN_ONSETS_FOR_EPOCH_AVERAGE
if not run_epoch_view:
    print(
        f"Stimulus-locked view skipped: {n_fitting} epoch(s) fit the {n_times}-sample "
        f"window, need {MIN_ONSETS_FOR_EPOCH_AVERAGE}. Raise N_TIMES_SUBSET (or use an "
        "experiment with stimulus annotations)."
    )
else:
    # epoch_average works on any array whose LAST axis is time, so the global
    # (K, F, T) maps go through unchanged.
    onset_tf, n_used = iva_quality.epoch_average(
        tf_maps, onset_samples_in, EPOCH_PRE, EPOCH_POST
    )
    epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq  # t = 0 at the onset
    EPOCH_MARKS = [0.0, min(AssrEpoch.STIMULUS_DURATION_S, float(epoch_times[-1]))]

    print(f"Onset-averaged TF maps : {onset_tf.shape}  (K, F, W)")
    print(
        f"Epochs averaged        : {n_used} of {len(onset_samples_in)} onset(s) in the "
        "window"
    )
    print(
        f"Epoch window           : {EPOCH_PRE + EPOCH_POST} samples ({EPOCH_PRE} pre, "
        f"{EPOCH_POST} post) = [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s"
    )
    if EPOCH_POST < int(round(EPOCH_POST_S * sfreq)):
        print(
            f"  NOTE: post-onset span trimmed from {EPOCH_POST_S} s to "
            f"{EPOCH_POST / sfreq:.3f} s by the shortest inter-onset gap."
        )
    # The baseline should read ~0; structure there is a warning sign, not a response.
    _baseline = np.abs(onset_tf[COMP_INDICES, :, :EPOCH_PRE]).mean()
    _response = np.abs(onset_tf[COMP_INDICES, :, EPOCH_PRE:]).mean()
    print(
        f"Mean |baseline| / mean |post-onset| : {_baseline:.4g} / {_response:.4g}  "
        f"(ratio {_baseline / _response:.2f})"
    )

    fig_tf_onset = plot_global_tf_grid(
        {SHARED_ROW: onset_tf},
        [SHARED_ROW],
        COMP_INDICES,
        FREQS,
        epoch_times,
        label=LABEL,
        title="Onset-averaged component TF maps",
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        sign_note=SIGN_NOTE,
        save_path=(PLOTS_DIR / "shared_tf_maps_onset.png") if SAVE_PLOTS else None,
    )
    plt.show()
    plt.close("all")